**Necessary Imports**

In [38]:
import pandas as pd 
import numpy as np 
import re 

**Loading the Necessary Files**

In [39]:
df = pd.read_csv("../Data/car.csv")
df.head()

,Name,Price,Used For,Transmisson,Colour,Make Year,Mileage,Engine (CC),Fuel,Kilometer Run,Waranty,Types
0,Hyundai | i20 Active S | TDi | 2015 | Hatchbac...,"रू. 24,75,000रू. 25,00,000",Private Use,Manual2WD,Brown,2015,14,1400,Petrol,42000,NaN,NaN
1,Excellent car on sale (Hyundai),"रू. 7,50,000",NaN,Auto2WD,Light blue,2005,11,1399,Petrol,87412,NaN,NaN
2,TATA 407 Container (Tata),"रू. 7,00,000",NaN,Manual - 2WD,White,2013,NaN,2956,Diesel,60000,NaN,NaN
3,4x4 swaraj Mazda (Mahindra),"रू. 6,00,000",NaN,Manual - 4WD,NaN,2017,NaN,NaN,Diesel,NaN,NaN,NaN
4,i20 Active good for used few time (Hyundai),रू. 375,NaN,Auto - 2WD,white,2019,17,1200,Petrol,2400,NaN,NaN


**Changing the columns into Snake Case**

In [40]:
df.columns = (
    df.columns               
      .str.lower()                  # convert to lowercase
      .str.replace('[^a-zA-Z0-9 ]', '', regex=True)   # remove special chars ()&./
      .str.replace(' +', '_', regex=True)  # convert spaces to underscores
)
df.head()

,name,price,used_for,transmisson,colour,make_year,mileage,engine_cc,fuel,kilometer_run,waranty,types
0,Hyundai | i20 Active S | TDi | 2015 | Hatchbac...,"रू. 24,75,000रू. 25,00,000",Private Use,Manual2WD,Brown,2015,14,1400,Petrol,42000,NaN,NaN
1,Excellent car on sale (Hyundai),"रू. 7,50,000",NaN,Auto2WD,Light blue,2005,11,1399,Petrol,87412,NaN,NaN
2,TATA 407 Container (Tata),"रू. 7,00,000",NaN,Manual - 2WD,White,2013,NaN,2956,Diesel,60000,NaN,NaN
3,4x4 swaraj Mazda (Mahindra),"रू. 6,00,000",NaN,Manual - 4WD,NaN,2017,NaN,NaN,Diesel,NaN,NaN,NaN
4,i20 Active good for used few time (Hyundai),रू. 375,NaN,Auto - 2WD,white,2019,17,1200,Petrol,2400,NaN,NaN


**Removing Currency Symbol**

In [41]:
df['price'] = df['price'].str.replace("रू.","",regex=False).str.replace(",","",regex=False).str.strip() #removes the rupee symbol and commas in the price. 
df.head()

,name,price,used_for,transmisson,colour,make_year,mileage,engine_cc,fuel,kilometer_run,waranty,types
0,Hyundai | i20 Active S | TDi | 2015 | Hatchbac...,2475000 2500000,Private Use,Manual2WD,Brown,2015,14,1400,Petrol,42000,NaN,NaN
1,Excellent car on sale (Hyundai),750000,NaN,Auto2WD,Light blue,2005,11,1399,Petrol,87412,NaN,NaN
2,TATA 407 Container (Tata),700000,NaN,Manual - 2WD,White,2013,NaN,2956,Diesel,60000,NaN,NaN
3,4x4 swaraj Mazda (Mahindra),600000,NaN,Manual - 4WD,NaN,2017,NaN,NaN,Diesel,NaN,NaN,NaN
4,i20 Active good for used few time (Hyundai),375,NaN,Auto - 2WD,white,2019,17,1200,Petrol,2400,NaN,NaN


There are some listings with name in Nepali but they are kept as it is because it is very difficult to extend semantic meaning from them. 

**Changing the transmission column into transmission_type and drive**

In [42]:
def split_transmisson(value):
    
    #first, let us check if the value is none. 
    if pd.isna(value):
        return (None,None)

    #Lets seperate by space
    value = value.strip()

    #If - is in the value then seperate by that value
    if '-' in value:
        parts = [p.strip() for p in value.split('-')]
        if len(parts) == 2:
            transmission_type = parts[0]
            drive = parts[1]
            return (transmission_type, drive)

    #Finally if the - doesn't exist we seperate alphabetic and numeric/WD parts 
    match = re.match(r"([A-Za-z]+)(.*)", value)
    if match:
        transmission_type = match.group(1)
        drive = match.group(2)
        return(transmission_type,drive)

    return(value,None)  

df[['transmisson_type','drive']] = df['transmisson'].apply(lambda x: pd.Series(split_transmisson(x)))
df.drop('transmisson', axis=1, inplace=True)
df.head() 


,name,price,used_for,colour,make_year,mileage,engine_cc,fuel,kilometer_run,waranty,types,transmisson_type,drive
0,Hyundai | i20 Active S | TDi | 2015 | Hatchbac...,2475000 2500000,Private Use,Brown,2015,14,1400,Petrol,42000,NaN,NaN,Manual,2WD
1,Excellent car on sale (Hyundai),750000,NaN,Light blue,2005,11,1399,Petrol,87412,NaN,NaN,Auto,2WD
2,TATA 407 Container (Tata),700000,NaN,White,2013,NaN,2956,Diesel,60000,NaN,NaN,Manual,2WD
3,4x4 swaraj Mazda (Mahindra),600000,NaN,NaN,2017,NaN,NaN,Diesel,NaN,NaN,NaN,Manual,4WD
4,i20 Active good for used few time (Hyundai),375,NaN,white,2019,17,1200,Petrol,2400,NaN,NaN,Auto,2WD


**Cleaning Numeric Columns make_year, mileage, engine_cc and kilometer_run **

In [44]:


def clean_numeric_columns(df, columns):
    for col in columns:
        df[col] = df[col].astype(str).str.extract(r'(\d+(?:\.\d+)?)')
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

df = clean_numeric_columns(df,['make_year','mileage','engine_cc','kilometer_run'])
df.head()


,name,price,used_for,colour,make_year,mileage,engine_cc,fuel,kilometer_run,waranty,types,transmisson_type,drive
0,Hyundai | i20 Active S | TDi | 2015 | Hatchbac...,2475000 2500000,Private Use,Brown,2015,14.0,1400.0,Petrol,42000.0,NaN,NaN,Manual,2WD
1,Excellent car on sale (Hyundai),750000,NaN,Light blue,2005,11.0,1399.0,Petrol,87412.0,NaN,NaN,Auto,2WD
2,TATA 407 Container (Tata),700000,NaN,White,2013,NaN,2956.0,Diesel,60000.0,NaN,NaN,Manual,2WD
3,4x4 swaraj Mazda (Mahindra),600000,NaN,NaN,2017,NaN,NaN,Diesel,NaN,NaN,NaN,Manual,4WD
4,i20 Active good for used few time (Hyundai),375,NaN,white,2019,17.0,1200.0,Petrol,2400.0,NaN,NaN,Auto,2WD
